# 30 — Generate Deteksi (YOLO fine-tune) untuk DUA Tracker

Kernel: `s2-main`.

Satu set deteksi dipakai **kedua** tracker (kesetaraan pembanding). Dua format:
- **DiffMOT**: `detections/{split}/{seq}/{frame:08d}.txt` — baris `frame,x,y,w,h,score` (kolom 1 dibuang kode DiffMOT)
- **OC-SORT**: `det_mot/{split}/{seq}.txt` — baris `frame,-1,x,y,w,h,score,-1,-1,-1`

Bobot = hasil Skenario A (`best.pt` dari Colab/GPU server). Taruh di `data/s2/weights/`.

In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

In [ ]:
# cek bobot yang tersedia
import glob
wts = sorted(glob.glob(str(DATA / "weights" / "*.pt")))
if not wts:
    raise SystemExit("Tidak ada bobot di data/s2/weights/ — download best.pt hasil Skenario A dari Colab/GPU server dulu")
for w in wts:
    print(w)

In [ ]:
# pilih bobot (default: yolo26n)
from pathlib import Path
WEIGHT = Path(input(f"Path bobot [{wts[0]}]: ").strip() or wts[0])
from ultralytics import YOLO
model = YOLO(str(WEIGHT))
print("model:", WEIGHT.name)

### Loop deteksi

Konfigurasi: `imgsz=640, conf=0.05, iou=0.7` — tulis SEMUA deteksi di atas 0.05; threshold
tinggi/rendah diserahkan ke tracker (DiffMOT high/low_thres; OC-SORT track_thresh).

In [ ]:
import cv2, time
import numpy as np
from pathlib import Path

def gen_detections(split_root: Path, det_dir: Path, det_mot_dir: Path, model, device=0, imgsz=640, conf=0.05, iou=0.7):
    det_dir.mkdir(parents=True, exist_ok=True)
    det_mot_dir.mkdir(parents=True, exist_ok=True)
    stats = []
    for seq in sorted(p for p in split_root.iterdir() if p.is_dir() and not p.name.endswith(".bad-old")):
        img_dir = seq / "img1"
        frames = sorted(img_dir.glob("*.*"))
        t0 = time.time(); tot_det = 0; tot_conf = 0.0
        mot_f = open(det_mot_dir / f"{seq.name}.txt", "w")
        for i, fp in enumerate(frames):
            frame = i + 1
            img = cv2.imread(str(fp))
            H, W = img.shape[:2]
            r = model.predict(img, imgsz=imgsz, conf=conf, iou=iou, device=device, verbose=False)[0]
            lines_d = []
            for b in r.boxes:
                x1, y1, x2, y2 = b.xyxy[0].tolist()
                x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
                w, h = x2 - x1, y2 - y1
                if w <= 1 or h <= 1:
                    continue
                sc = float(b.conf)
                lines_d.append(f"{frame},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{sc:.4f}\n")
                mot_f.write(f"{frame},-1,{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{sc:.4f},-1,-1,-1\n")
            if lines_d:
                (det_dir / seq.name).mkdir(parents=True, exist_ok=True)
                (det_dir / seq.name / f"{frame:08d}.txt").write_text("".join(lines_d))
            tot_det += len(lines_d); tot_conf += sum(float(l.split(",")[5]) for l in lines_d)
        mot_f.close()
        n = len(frames); dt = time.time() - t0
        stats.append({"seq": seq.name, "frames": n, "dets": tot_det,
                      "mean_conf": (tot_conf / tot_det) if tot_det else 0.0,
                      "det_per_frame": (tot_det / n) if n else 0.0, "seconds": round(dt, 1)})
        print(f"{seq.name:16s} frames={n:5d} dets={tot_det:6d} {dt:6.1f}s")
    return stats

import pandas as pd
stats_all = []
for ds, split in [("mot20", "train"), ("dancetrack", "val")]:
    split_root = DATA / ds / split
    if not split_root.exists():
        print("!! skip", split_root); continue
    print("\n=== ", ds, split, "===")
    st = gen_detections(
        split_root,
        DATA / ds / "detections" / split,     # format DiffMOT (per-frame)
        DATA / ds / "det_mot" / split,        # format OC-SORT (det.txt per seq)
        model,
    )
    stats_all += st
df = pd.DataFrame(stats_all)
df.to_csv(EXP / "detection_stats.csv", index=False)
display(df)
print("\nsaved:", EXP / "detection_stats.csv")

### (Opsional) Deteksi resmi DiffMOT untuk sanity check

`Detections.zip` (release v1.1, 272 MB) dipakai untuk memverifikasi reproduksi angka publik
DiffMOT. Tidak wajib — hanya untuk sanity.

In [ ]:
# opsional
!wget -qO $S2_DATA/diffmot_detections.zip https://github.com/Kroery/DiffMOT/releases/download/v1.1/Detections.zip
!mkdir -p $S2_DATA/diffmot_detections && unzip -q $S2_DATA/diffmot_detections.zip -d $S2_DATA/diffmot_detections
!find $S2_DATA/diffmot_detections -maxdepth 3 -type d | head -20

**Lanjut**: `40_s2_diffmot_embeddings.ipynb` (kernel `s2-diffmot`) — patch + smoke ReID.